# Calvarial Bone/Mineralization Analysis

This notebook reproduces the bilateral parietal/calvarial VOI workflow used for the juvenile mouse skull µCT analysis.

## Endpoints exported
- TV
- BV
- BV/TV
- HU-based TMD
- Apparent HU-based BMD
- BMC index

## Important interpretation note
BV/TV is derived from thresholded voxels in a fixed, surface-following bilateral parietal/calvarial VOI.

TMD and apparent BMD are HU-based relative metrics, not true hydroxyapatite-calibrated density values. BMC is a relative HU·mm³ index unless an HA phantom calibration is added.


## How to use

1. Put your `.ct.zip` scan file in an accessible folder.
2. Update the variables in the **Run analysis** cell:
   - `input_zip`
   - `sample_id`
   - `output_dir`
3. Run the notebook cells in order.

For shifted scans, you can manually set `manual_y_search = (300, 500)`. Otherwise, leave it as `None` and the notebook will choose the default or adjusted y-search window automatically.


In [1]:
#!/usr/bin/env python3
"""
Calvarial bone/mineralization analysis for Bruker/CT .ct.zip skull scans.

This script reproduces the bilateral parietal/calvarial VOI workflow used.

Endpoints exported:
    - TV_mm3
    - BV_mm3
    - BV_TV
    - BV_TV_percent
    - TMD_HU
    - BMD_HU_apparent
    - BMC_index_HU_mm3

Important interpretation:
    BV/TV is derived from thresholded voxels in a fixed, surface-following
    bilateral parietal/calvarial VOI.

    TMD_HU and BMD_HU_apparent are HU-based relative metrics, not true
    hydroxyapatite-calibrated density values.

    BMC_index_HU_mm3 is a relative HU*mm3 index, not true mineral content,
    unless an HA phantom calibration is added separately.

Example:
    python calvarial_bone_mineralization_analysis.py \
        --input "sample.ct.zip" \
        --sample-id "R5(7)" \
        --output-dir "./outputs"

Optional:
    python calvarial_bone_mineralization_analysis.py \
        --input "sample.ct.zip" \
        --sample-id "B5(4)" \
        --output-dir "./outputs" \
        --y-search 300 500
"""

from __future__ import annotations

import argparse
import csv
import re
import zipfile
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import scipy.ndimage as ndi
from PIL import Image, ImageDraw, ImageFont


# -----------------------------
# Default analysis parameters
# -----------------------------

DEFAULT_SHAPE = (1008, 512, 512)  # z, y, x
DEFAULT_DTYPE = np.int16
DEFAULT_VOXEL_MM = 0.046922

# Fixed bilateral parietal/calvarial VOI used in this analysis.
Z_START = 520
Z_STOP = 600
X_RANGES = {
    "Left_parietal": (175, 255),
    "Right_parietal": (275, 355),
}

# Surface detection and VOI band.
DETECT_THR_HU = 150
DEFAULT_Y_SEARCH = (70, 220)
ADJUSTED_Y_SEARCH = (300, 500)
BAND_UP = 4
BAND_DOWN = 22

# Threshold sensitivity and primary threshold.
THRESHOLDS = [250, 350, 450]
PRIMARY_THR = 350


def make_safe_id(sample_id: str) -> str:
    """Convert sample IDs such as 'R5(7)' to filename-safe 'R5_7'."""
    safe = sample_id.replace("(", "_").replace(")", "")
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", safe)
    safe = re.sub(r"_+", "_", safe).strip("_")
    return safe


def extract_img_from_ct_zip(zip_path: Path, work_dir: Path) -> Path:
    """Extract the first .img file from a Bruker/CT .ct.zip archive."""
    work_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        img_candidates = [n for n in z.namelist() if n.lower().endswith(".img")]
        if not img_candidates:
            raise FileNotFoundError(f"No .img file found in {zip_path}")

        img_name = img_candidates[0]
        img_path = work_dir / Path(img_name).name

        if not img_path.exists():
            z.extract(img_name, work_dir)
            extracted = work_dir / img_name

            # If the archive stores nested folders, move extracted IMG to work_dir root.
            if extracted != img_path and extracted.exists():
                extracted.rename(img_path)

    return img_path


def load_volume(
    img_path: Path,
    shape: Tuple[int, int, int] = DEFAULT_SHAPE,
    dtype=DEFAULT_DTYPE,
) -> np.memmap:
    """Memory-map the CT volume."""
    return np.memmap(img_path, dtype=dtype, mode="r", shape=shape)


def count_surface_detections(
    arr: np.memmap,
    y_search: Tuple[int, int],
    x_ranges: Dict[str, Tuple[int, int]] = X_RANGES,
    z_start: int = Z_START,
    z_stop: int = Z_STOP,
    detect_thr_hu: int = DETECT_THR_HU,
) -> Tuple[int, int]:
    """
    Count how many x-columns contain a detectable bone surface in the y-search window.

    This is used to decide whether a shifted scan needs the adjusted y-window.
    """
    total_detected = 0
    total_cols = 0

    for _, (x0, x1) in x_ranges.items():
        for z in range(z_start, z_stop):
            slab = np.asarray(arr[z, y_search[0]:y_search[1], x0:x1])
            total_detected += int((slab > detect_thr_hu).any(axis=0).sum())
            total_cols += x1 - x0

    return total_detected, total_cols


def choose_y_search(
    arr: np.memmap,
    forced_y_search: Tuple[int, int] | None = None,
) -> Tuple[Tuple[int, int], str]:
    """
    Choose the surface-search window.

    Most scans in this batch used y=70-220. A few shifted scans required
    y=300-500 to identify the same calvarial surface.
    """
    if forced_y_search is not None:
        y0, y1 = forced_y_search
        return (y0, y1), f"User-specified y-search window {y0}-{y1} was used."

    det_default, total_cols = count_surface_detections(arr, DEFAULT_Y_SEARCH)
    det_adjusted, _ = count_surface_detections(arr, ADJUSTED_Y_SEARCH)

    if det_default < 0.5 * total_cols and det_adjusted > det_default:
        return (
            ADJUSTED_Y_SEARCH,
            "For this scan, the y-search window was adjusted to 300-500 because the skull was shifted relative to prior samples.",
        )

    return DEFAULT_Y_SEARCH, "Standard y-search window 70-220 was used."


def build_surface(
    arr: np.memmap,
    x0: int,
    x1: int,
    y_search: Tuple[int, int],
    z_start: int = Z_START,
    z_stop: int = Z_STOP,
    detect_thr_hu: int = DETECT_THR_HU,
) -> np.ndarray:
    """
    Detect the outer calvarial surface.

    For each z/x column, the surface is defined as the first y-position within
    the search window with intensity > detect_thr_hu. Missing values are filled
    by nearest-neighbor distance transform and median-filtered.
    """
    nz, nx = z_stop - z_start, x1 - x0
    surf = np.full((nz, nx), np.nan, dtype=float)

    for iz, z in enumerate(range(z_start, z_stop)):
        slab = np.asarray(arr[z, y_search[0]:y_search[1], x0:x1])
        for ix in range(nx):
            ys = np.where(slab[:, ix] > detect_thr_hu)[0]
            if len(ys):
                surf[iz, ix] = y_search[0] + ys[0]

    valid = np.isfinite(surf)
    if not valid.any():
        raise ValueError(
            "No surface detected in the selected y-search window. "
            "Try --y-search 300 500 or manually inspect the scan orientation."
        )

    inds = ndi.distance_transform_edt(
        ~valid,
        return_distances=False,
        return_indices=True,
    )
    filled = surf[tuple(inds)]
    return ndi.median_filter(filled, size=(5, 5), mode="nearest")


def extract_voi_values(
    arr: np.memmap,
    x0: int,
    x1: int,
    surface: np.ndarray,
    z_start: int = Z_START,
    z_stop: int = Z_STOP,
    band_up: int = BAND_UP,
    band_down: int = BAND_DOWN,
) -> np.ndarray:
    """
    Extract values from a surface-following VOI band.

    For each detected surface point, the VOI extends from:
        y = surface - band_up
    to:
        y = surface + band_down
    inclusive.
    """
    vals = []

    for iz, z in enumerate(range(z_start, z_stop)):
        for ix, x in enumerate(range(x0, x1)):
            yc = int(round(surface[iz, ix]))
            y0 = max(0, yc - band_up)
            y1 = min(arr.shape[1], yc + band_down + 1)
            vals.append(np.asarray(arr[z, y0:y1, x]))

    return np.concatenate(vals)


def calculate_metrics(
    sample_id: str,
    voi_values: Dict[str, np.ndarray],
    thresholds: Iterable[int],
    voxel_vol_mm3: float,
    y_search: Tuple[int, int],
) -> List[dict]:
    """Calculate regional and pooled mineralization metrics."""
    rows = []

    for thr in thresholds:
        pooled = []

        for region, vals in voi_values.items():
            bone = vals > thr
            tv_mm3 = vals.size * voxel_vol_mm3
            bv_mm3 = int(bone.sum()) * voxel_vol_mm3
            bv_tv = bv_mm3 / tv_mm3 if tv_mm3 else np.nan
            tmd_hu = float(vals[bone].mean()) if bone.any() else np.nan
            bmc_index = float(vals[bone].sum() * voxel_vol_mm3) if bone.any() else np.nan
            bmd_hu = bmc_index / tv_mm3 if tv_mm3 else np.nan

            rows.append(
                {
                    "Sample_ID": sample_id,
                    "Region": region,
                    "Threshold_HU": thr,
                    "TV_mm3": tv_mm3,
                    "BV_mm3": bv_mm3,
                    "BV_TV": bv_tv,
                    "BV_TV_percent": bv_tv * 100,
                    "TMD_HU": tmd_hu,
                    "BMD_HU_apparent": bmd_hu,
                    "BMC_index_HU_mm3": bmc_index,
                    "Surface_search_window_y": f"{y_search[0]}-{y_search[1]}",
                }
            )

            pooled.append(vals)

        vals = np.concatenate(pooled)
        bone = vals > thr
        tv_mm3 = vals.size * voxel_vol_mm3
        bv_mm3 = int(bone.sum()) * voxel_vol_mm3
        bv_tv = bv_mm3 / tv_mm3 if tv_mm3 else np.nan
        tmd_hu = float(vals[bone].mean()) if bone.any() else np.nan
        bmc_index = float(vals[bone].sum() * voxel_vol_mm3) if bone.any() else np.nan
        bmd_hu = bmc_index / tv_mm3 if tv_mm3 else np.nan

        rows.append(
            {
                "Sample_ID": sample_id,
                "Region": "Bilateral_parietal_pooled",
                "Threshold_HU": thr,
                "TV_mm3": tv_mm3,
                "BV_mm3": bv_mm3,
                "BV_TV": bv_tv,
                "BV_TV_percent": bv_tv * 100,
                "TMD_HU": tmd_hu,
                "BMD_HU_apparent": bmd_hu,
                "BMC_index_HU_mm3": bmc_index,
                "Surface_search_window_y": f"{y_search[0]}-{y_search[1]}",
            }
        )

    return rows


def save_csvs(
    rows: List[dict],
    output_dir: Path,
    safe_id: str,
    primary_thr: int,
    y_note: str,
) -> Tuple[Path, Path]:
    """Save threshold-sensitivity CSV and primary-threshold CSV."""
    output_dir.mkdir(parents=True, exist_ok=True)

    sensitivity_csv = output_dir / f"{safe_id}_calvarial_threshold_sensitivity.csv"
    with open(sensitivity_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)

    primary_rows = [r for r in rows if r["Threshold_HU"] == primary_thr]
    primary_csv = output_dir / f"{safe_id}_calvarial_mineralization_metrics.csv"

    with open(primary_csv, "w", newline="") as f:
        fieldnames = list(primary_rows[0].keys()) + ["Interpretation_note"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for r in primary_rows:
            rr = dict(r)
            rr["Interpretation_note"] = (
                "BV/TV from a surface-following bilateral parietal VOI. "
                "TMD and apparent BMD are HU-based, not mg HA/cm3. "
                "BMC is a relative HU*mm3 index because no HA phantom calibration was provided. "
                + y_note
            )
            writer.writerow(rr)

    return primary_csv, sensitivity_csv


def overlay_slice(
    arr: np.memmap,
    z: int,
    surfaces: Dict[str, np.ndarray],
    x_ranges: Dict[str, Tuple[int, int]],
    primary_thr: int = PRIMARY_THR,
    z_start: int = Z_START,
    band_up: int = BAND_UP,
    band_down: int = BAND_DOWN,
    annotated: bool = False,
) -> Image.Image:
    """Create a grayscale coronal slice with VOI boundaries and thresholded bone overlay."""
    sl = np.asarray(arr[z])
    base = (np.clip((sl + 1000) / 3500, 0, 1) * 255).astype(np.uint8)
    rgb = np.stack([base, base, base], axis=-1)

    iz = z - z_start
    for region, (x0, x1) in x_ranges.items():
        surf = surfaces[region]
        for ix, x in enumerate(range(x0, x1)):
            yc = int(round(surf[iz, ix]))
            y0 = max(0, yc - band_up)
            y1 = min(arr.shape[1], yc + band_down + 1)

            if ix in (0, x1 - x0 - 1):
                rgb[y0:y1, x, :] = [255, 255, 255]

            step = 2 if annotated else 4
            if x % step == 0:
                rgb[y0, x, :] = [255, 255, 255]
                rgb[y1 - 1, x, :] = [255, 255, 255]

            ys = np.arange(y0, y1)
            bone_y = ys[sl[y0:y1, x] > primary_thr]
            rgb[bone_y, x, :] = [255, 60, 60]

    return Image.fromarray(rgb)


def save_qc_images(
    arr: np.memmap,
    surfaces: Dict[str, np.ndarray],
    output_dir: Path,
    safe_id: str,
    sample_id: str,
    y_search: Tuple[int, int],
    adjusted_y_search: Tuple[int, int] = ADJUSTED_Y_SEARCH,
) -> Tuple[Path, Path]:
    """Save QC montage and representative annotated scan."""
    output_dir.mkdir(parents=True, exist_ok=True)

    # Four-slice QC montage.
    tiles = []
    for z in [520, 540, 560, 580]:
        im = overlay_slice(arr, z, surfaces, X_RANGES)
        d = ImageDraw.Draw(im)
        d.rectangle((8, 8, 150, 38), fill="black")
        d.text((14, 14), f"z = {z}", fill="white")
        tiles.append(im)

    montage = Image.new("RGB", (1024, 1024), "black")
    for i, tile in enumerate(tiles):
        montage.paste(tile, ((i % 2) * 512, (i // 2) * 512))

    qc_path = output_dir / f"{safe_id}_calvarial_ROI_QC.png"
    montage.save(qc_path)

    # Representative annotated coronal slice.
    rep_z = 560
    rep = overlay_slice(arr, rep_z, surfaces, X_RANGES, annotated=True)
    draw = ImageDraw.Draw(rep)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 18)
        fontb = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
    except Exception:
        font = ImageFont.load_default()
        fontb = ImageFont.load_default()

    shifted = y_search == adjusted_y_search
    box_h = 137 if shifted else 115
    draw.rectangle((10, 10, 505, box_h), fill=(0, 0, 0))
    draw.text((18, 18), f"{sample_id} representative coronal slice (z={rep_z})", fill="white", font=fontb)
    draw.text((18, 48), "White = bilateral parietal VOI boundary", fill="white", font=font)
    draw.text((18, 70), f"Red = voxels > {PRIMARY_THR} HU within VOI", fill="white", font=font)
    draw.text((18, 92), "VOI follows outer calvarial surface and excludes midline suture", fill="white", font=font)

    if shifted:
        draw.text((18, 114), "Note: y-search adjusted because skull is shifted in this scan", fill="white", font=font)
        label_y = 365
        left_line = (135, 375, 200, 355)
        right_line = (380, 375, 320, 355)
        mid_line = (255, 390, 255, 445)
        mid_text = (150, 448)
    else:
        label_y = 118
        left_line = (135, 125, 200, 145)
        right_line = (380, 125, 320, 145)
        mid_line = (255, 160, 255, 210)
        mid_text = (150, 212)

    draw.line(left_line, fill=(0, 255, 255), width=3)
    draw.text((15, label_y), "Left parietal VOI", fill=(0, 255, 255), font=fontb)
    draw.line(right_line, fill=(0, 255, 255), width=3)
    draw.text((305, label_y), "Right parietal VOI", fill=(0, 255, 255), font=fontb)
    draw.line(mid_line, fill=(255, 255, 0), width=3)
    draw.text(mid_text, "Midline sagittal suture excluded", fill=(255, 255, 0), font=font)

    annotated_path = output_dir / f"{safe_id}_calvarial_representative_annotated.png"
    rep.save(annotated_path)

    return qc_path, annotated_path


def analyze_scan(
    input_zip: Path,
    sample_id: str,
    output_dir: Path,
    y_search: Tuple[int, int] | None = None,
    shape: Tuple[int, int, int] = DEFAULT_SHAPE,
    voxel_mm: float = DEFAULT_VOXEL_MM,
) -> dict:
    """Run full calvarial bone/mineralization analysis on a single CT zip."""
    safe_id = make_safe_id(sample_id)
    work_dir = output_dir / f"{safe_id}_bone_mineralization_work"
    img_path = extract_img_from_ct_zip(input_zip, work_dir)
    arr = load_volume(img_path, shape=shape)

    selected_y_search, y_note = choose_y_search(arr, forced_y_search=y_search)
    voxel_vol_mm3 = voxel_mm ** 3

    surfaces = {
        name: build_surface(arr, *xr, selected_y_search)
        for name, xr in X_RANGES.items()
    }

    voi_values = {
        name: extract_voi_values(arr, *X_RANGES[name], surfaces[name])
        for name in X_RANGES
    }

    rows = calculate_metrics(
        sample_id=sample_id,
        voi_values=voi_values,
        thresholds=THRESHOLDS,
        voxel_vol_mm3=voxel_vol_mm3,
        y_search=selected_y_search,
    )

    primary_csv, sensitivity_csv = save_csvs(
        rows=rows,
        output_dir=output_dir,
        safe_id=safe_id,
        primary_thr=PRIMARY_THR,
        y_note=y_note,
    )

    qc_path, annotated_path = save_qc_images(
        arr=arr,
        surfaces=surfaces,
        output_dir=output_dir,
        safe_id=safe_id,
        sample_id=sample_id,
        y_search=selected_y_search,
    )

    primary_pooled = [
        r for r in rows
        if r["Threshold_HU"] == PRIMARY_THR
        and r["Region"] == "Bilateral_parietal_pooled"
    ][0]

    return {
        "sample_id": sample_id,
        "safe_id": safe_id,
        "primary_csv": primary_csv,
        "sensitivity_csv": sensitivity_csv,
        "qc_path": qc_path,
        "annotated_path": annotated_path,
        "primary_pooled": primary_pooled,
        "y_search": selected_y_search,
        "y_note": y_note,
    }


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Run bilateral parietal/calvarial VOI bone/mineralization analysis on a Bruker CT .ct.zip file."
    )
    parser.add_argument(
        "--input",
        required=True,
        type=Path,
        help="Path to Bruker/CT .ct.zip file containing a reconstructed .img volume.",
    )
    parser.add_argument(
        "--sample-id",
        required=True,
        help='Sample identifier to write into CSVs, e.g. "R5(7)".',
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=Path("."),
        help="Directory for output CSVs and QC images.",
    )
    parser.add_argument(
        "--voxel-mm",
        type=float,
        default=DEFAULT_VOXEL_MM,
        help=f"Voxel size in mm. Default: {DEFAULT_VOXEL_MM}",
    )
    parser.add_argument(
        "--shape",
        nargs=3,
        type=int,
        default=DEFAULT_SHAPE,
        metavar=("Z", "Y", "X"),
        help=f"Volume shape. Default: {DEFAULT_SHAPE}",
    )
    parser.add_argument(
        "--y-search",
        nargs=2,
        type=int,
        default=None,
        metavar=("Y0", "Y1"),
        help=(
            "Optional manual y-search window for surface detection. "
            "Usually 70 220; shifted scans may need 300 500."
        ),
    )
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    result = analyze_scan(
        input_zip=args.input,
        sample_id=args.sample_id,
        output_dir=args.output_dir,
        y_search=tuple(args.y_search) if args.y_search is not None else None,
        shape=tuple(args.shape),
        voxel_mm=args.voxel_mm,
    )

    pooled = result["primary_pooled"]
    print(f"Done: {result['sample_id']}")
    print(f"Surface search window y: {result['y_search'][0]}-{result['y_search'][1]}")
    print("\nPrimary 350-HU pooled bilateral parietal result:")
    for key in [
        "TV_mm3",
        "BV_mm3",
        "BV_TV",
        "BV_TV_percent",
        "TMD_HU",
        "BMD_HU_apparent",
        "BMC_index_HU_mm3",
    ]:
        print(f"{key}: {pooled[key]:.6g}")

    print("\nOutput files:")
    print(result["primary_csv"])
    print(result["sensitivity_csv"])
    print(result["qc_path"])
    print(result["annotated_path"])




In [2]:
# -----------------------------
# Run analysis
# -----------------------------
from pathlib import Path

# Update these three values for each sample.
input_zip = Path("/mnt/data/example_scan.ct.zip")  # <-- replace with your .ct.zip path
sample_id = "Sample_ID"                           # <-- e.g., "R5(7)"
output_dir = Path("/mnt/data/calvarial_outputs")  # <-- change as needed

# Optional manual override:
#   None = automatically choose default y-search or adjusted shifted-scan window
#   (70, 220) = standard search window
#   (300, 500) = shifted-scan search window
manual_y_search = None

result = analyze_scan(
    input_zip=input_zip,
    sample_id=sample_id,
    output_dir=output_dir,
    y_search=manual_y_search,
)

pooled = result["primary_pooled"]

print(f"Done: {result['sample_id']}")
print(f"Surface search window y: {result['y_search'][0]}-{result['y_search'][1]}")
print("\nPrimary 350-HU pooled bilateral parietal result:")
for key in [
    "TV_mm3",
    "BV_mm3",
    "BV_TV",
    "BV_TV_percent",
    "TMD_HU",
    "BMD_HU_apparent",
    "BMC_index_HU_mm3",
]:
    print(f"{key}: {pooled[key]:.6g}")

print("\nOutput files:")
print(result["primary_csv"])
print(result["sensitivity_csv"])
print(result["qc_path"])
print(result["annotated_path"])


FileNotFoundError: [Errno 2] No such file or directory: '\\mnt\\data\\example_scan.ct.zip'

## Batch use

To process multiple samples, repeat the **Run analysis** cell with a new `input_zip` and `sample_id`, or adapt the cell into a loop over a table of sample IDs and file paths.
